In [1]:
import os
import numpy as np
import pandas as pd

import yfinance as yf
import pandas_datareader as pdr

### Download Monthly ETF Prices

In [2]:
DATA_DIR   = './data/'
START_DATE = '2004-12-01' # extra month so we can calculate return in Jan 2005
END_DATE   = '2026-04-30'

# choose a few ETFs to represent assets a retail investor can access
TICKERS = ['SPY',  # US large cap
           'IWM',  # US small cap
           'QQQ',  # US tech heavy
           'EFA',  # Developed international
           'EEM',  # Emerging markets
           'TLT',  # Long-term Treasuries
           'TIP',  # TIPS (Treasury Inflation-Protected securities)
           'LQD',  # Investment grade corporate bonds
           'GLD',  # Gold
           'VNQ',  # US REITs (Real Estate Investment Trusts)
           ]

In [3]:
print('Downloading daily adjusted close prices...')
raw = yf.download(tickers     = TICKERS,
                  start       = START_DATE,
                  end         = END_DATE,
                  auto_adjust = True, # adjusts for splits and dividends
                  progress    = True,
                  )['Close']

monthly_prices = raw.resample('ME').last() # resample for month-end

print(f'\nDate range:     {monthly_prices.index[0].date()} to {monthly_prices.index[-1].date()}')
print(f'Number of months: {len(monthly_prices)}')
print(f'\nFirst available date per ticker:')
print(monthly_prices.apply(lambda col: col.first_valid_index()).to_string())

monthly_prices.head()

[*********************100%***********************]  10 of 10 completed


Date range:     2004-12-31 to 2026-04-30
Number of months: 257

First available date per ticker:
Ticker
EEM   2004-12-31
EFA   2004-12-31
GLD   2004-12-31
IWM   2004-12-31
LQD   2004-12-31
QQQ   2004-12-31
SPY   2004-12-31
TIP   2004-12-31
TLT   2004-12-31
VNQ   2004-12-31


Ticker,EEM,EFA,GLD,IWM,LQD,QQQ,SPY,TIP,TLT,VNQ
Date,,,,,,,,,,
2004-12-31,14.639677,28.706865,43.799999,48.706837,47.045006,33.973446,81.559212,54.419128,44.138424,22.614826
2005-01-31,14.562074,28.160486,42.220001,46.728481,47.622856,31.828825,79.730637,54.424324,45.713554,20.679266
2005-02-28,15.972010,29.226366,43.529999,47.495762,47.177006,31.675640,81.397278,54.136887,45.040592,21.335119
2005-03-31,14.708582,28.459663,42.820000,46.149357,46.581524,31.122459,79.908279,54.255344,44.835800,20.970825
2005-04-30,14.525087,27.999268,43.349998,43.538696,47.336887,29.769321,78.411179,55.219727,46.568790,22.180840


In [4]:
output_path = os.path.join(DATA_DIR, 'monthly-etf-prices.parquet')
monthly_prices.to_parquet(output_path)
print(f'\nSaved ETF price data to {output_path}')


Saved ETF price data to ./data/monthly-etf-prices.parquet


### Get Risk-Free-Rate Data
Pull risk-free-rate from the Fama-French monthly factor dataset

In [26]:
# pull the fama-french 3 factor dataset
ff_factors = pdr.get_data_famafrench('F-F_Research_Data_Factors', start='2005-01-01')[0]
rf         = pd.DataFrame(ff_factors['RF'] / 100.)  # convert from percent to decimal

# convert index to datetime and add the month-end day to match the yfinance data
rf.index = pd.to_datetime(rf.index.astype(str), format='%Y-%m') + pd.offsets.MonthEnd(0)

rf.head()

/tmp/ipykernel_50677/414501489.py:2: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_factors = pdr.get_data_famafrench('F-F_Research_Data_Factors', start='2005-01-01')[0]
/tmp/ipykernel_50677/414501489.py:2: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_factors = pdr.get_data_famafrench('F-F_Research_Data_Factors', start='2005-01-01')[0]


,RF
Date,
2005-01-31,0.0016
2005-02-28,0.0016
2005-03-31,0.0021
2005-04-30,0.0021
2005-05-31,0.0024


In [5]:
output_path = os.path.join(DATA_DIR, 'monthly-rfr.parquet')
monthly_prices.to_parquet(output_path)
print(f'\nSaved risk-free-rate data to {output_path}')


Saved risk-free-rate data to ./data/monthly-rfr.parquet
